# Méta-classes

## Implémentation du pattern singleton

Le pattern singleton consiste en le fait de renvoyer toujours la même instance quand du code client demande une nouvelle instance d'une classe.

Créez une méta-classe `Singleton` qui implémente ce pattern. Attention, notez bien qu'une méta-classe peut être appliquée à plusieurs classes.

Une méta-classe peut ré-implémeter plusieurs méthodes. Ici, le plus simple sera de redéfinir `__call__`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
class SingletonMeta(type):

  _instances = {}

  def __call__(cls, *args, **kwargs):
    if cls not in SingletonMeta._instances:
      SingletonMeta._instances[cls] = super().__call__(*args, **kwargs)
    return SingletonMeta._instances[cls]


class A(metaclass=SingletonMeta):
    pass


a1 = A()
a2 = A()
print(a1 is a2)

## Modification de méthodes

Écrivez une métaclasse qui transforme toutes les méthodes contenant le mot "compute" d'une classe avec le décorateur suivant, qui log les appels aux méthodes qu'il décore :

In [ ]:
import functools


def log_calls(f):

  @functools.wraps(f)
  def wrapper(self, *args, **kwargs):
    result = f(self, *args, **kwargs)
    print(f"{f.__name__} appelée avec les args {args} et kwargs {kwargs} a "
          f"calculé {result}")
    return result

  return wrapper

In [ ]:
# Votre code ici

### Solution

In [ ]:
class LogComputeMeta(type):
  def __new__(self, name, bases, dct):
    new_dct = {k: (log_calls(v) if "compute" in k else v) for k, v in dct.items()}
    return super().__new__(self, name, bases, new_dct)

class A(metaclass=LogComputeMeta):
  def compute_addition(self, a, b):
    return a + b

a = A()
a.compute_addition(1, 2)

## Modification de la hierarchie de classes

Il est aussi possible avec des méta-classes de modifier les parents d'une classe.

Créez une méta-classe qui ajoute la classe `DataSaver` aux parents de la classe produite si celle-ci contient un attribut `data`.

In [ ]:
class DataSaver:
  def save(self):
    print(f"Saving data {self.data}…")


# Votre code ici

### Solution

In [ ]:
class DataSaverMeta(type):
  def __new__(self, name, bases, dct):
    if "data" in dct:
      bases += (DataSaver,)
    return super().__new__(self, name, bases, dct)


class A(metaclass=DataSaverMeta):
  data = [1, 2]


class B(metaclass=DataSaverMeta):
  a = [1, 2]


a = A()
b = B()
a.save()
try:
    b.save()
except AttributeError as error:
    print("B n'a pas d'attribut data, donc pas de DataSaver parmi ses parents :", error)

## Valider les sous-classes&nbsp;: méta-classe ou `__init_subclass__`

On veut que toute sous-classe d'`Exporter` définisse un attribut de classe `extension`, une chaîne commençant par un point, et qu'une `TypeError` soit levée dès la définition de la classe sinon.

1. Implémentez cette vérification avec une méta-classe `ExporterMeta`.
2. Implémentez-la ensuite sans méta-classe, avec `__init_subclass__`.

Dans les deux cas, `class CsvExporter(Exporter): extension = ".csv"` doit être acceptée, et `class BrokenExporter(Exporter): pass` doit lever une `TypeError`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
class ExporterMeta(type):
    def __init__(cls, name, bases, namespace):
        super().__init__(name, bases, namespace)
        if bases and not str(getattr(cls, "extension", "")).startswith("."):
            raise TypeError(f"{cls.__name__} doit définir une extension commençant par un point")


class Exporter(metaclass=ExporterMeta):
    pass


class CsvExporter(Exporter):
    extension = ".csv"


try:

    class BrokenExporter(Exporter):
        pass

except TypeError as error:
    print(error)


# Sans méta-classe
class Exporter:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        if not str(getattr(cls, "extension", "")).startswith("."):
            raise TypeError(f"{cls.__name__} doit définir une extension commençant par un point")


class CsvExporter(Exporter):
    extension = ".csv"


try:

    class BrokenExporter(Exporter):
        pass

except TypeError as error:
    print(error)

`__init_subclass__` donne le même résultat sans méta-classe&nbsp;: le code est plus simple, et la classe reste combinable avec d'autres classes qui ont leur propre méta-classe.